# Context-Deference — MVP driver

Produces the **decisive first output**: the pairwise-cosine matrix across the three per-behavior
*suppression* directions — **raw**, then **residual** (after projecting out each behavior's signal
direction). Shared subspace? Orthogonal? Something in between?

Requires a GPU (Colab Pro A100). `src/` is imported, not reimplemented here.

> ⚠️ Two design decisions (contrast + behavioral object) are read from `configs/behaviors.yaml`
> and printed below. They are **recommended defaults pending a human call** — see the README.

In [1]:
# --- setup ---
# On Colab, install first:  !pip install -q -r ../requirements.txt
import sys, os, json
sys.path.append(os.path.abspath(".."))   # repo root, so `import src...` resolves
import numpy as np
import torch
import matplotlib.pyplot as plt

from src import model as M, data as D, directions as Dir, projection as P
from src import subspace as S, pipeline as PL

torch.set_grad_enabled(False)

/Users/childrebelsoldier/suppression/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


torch.autograd.grad_mode.set_grad_enabled(mode=False)

In [2]:
# --- config + the two design decisions ---
# Env overrides let one notebook serve both the science run (defaults: Llama-8B, full data, on a GPU)
# and a fast local validation (e.g. CD_MODEL=qwen2.5-1.5b-instruct CD_MAX_PAIRS=16 CD_MAX_NEW_TOKENS=24).
import os
cfg        = D.load_behaviors_config("../configs/behaviors.yaml")
models_cfg = D.load_yaml("../configs/models.yaml")

CONTRAST_MODE   = cfg["contrast"]["mode"]
BEHAVIORAL_MODE = cfg["behavioral_object"]["mode"]
SUBSET          = os.environ["CD_SUBSET"].split(",") if os.environ.get("CD_SUBSET") else cfg["mvp_subset"]
MAX_PAIRS       = int(os.environ.get("CD_MAX_PAIRS", "0")) or None      # None = full data
MAX_NEW_TOKENS  = int(os.environ.get("CD_MAX_NEW_TOKENS", "64"))

mc = models_cfg["models"][os.environ.get("CD_MODEL", models_cfg["default_model"])]
LAYER, POS = mc["default_layer"], mc["default_position"]

print("model            :", mc["tl_name"])
print("layer / position :", LAYER, "/", POS)
print("behaviors        :", SUBSET, "| max_pairs:", MAX_PAIRS, "| max_new_tokens:", MAX_NEW_TOKENS)
print("contrast.mode    :", CONTRAST_MODE, "  <-- DESIGN DECISION 1 (pending)")
print("behavioral.mode  :", BEHAVIORAL_MODE, "  <-- DESIGN DECISION 2 (confirmed)")

model            : Qwen/Qwen2.5-1.5B-Instruct
layer / position : 14 / -1
behaviors        : ['sycophancy', 'safety', 'knowledge_conflict'] | max_pairs: 8 | max_new_tokens: 10
contrast.mode    : overrode_vs_resisted   <-- DESIGN DECISION 1 (pending)
behavioral.mode  : correctness_label   <-- DESIGN DECISION 2 (confirmed)


In [3]:
# --- load model (GPU) ---
bundle = M.load_model(mc["tl_name"], dtype=mc.get("dtype", "bfloat16"))
print("loaded", bundle.name, "| layers:", bundle.n_layers,
      "| d_model:", bundle.d_model, "| device:", bundle.device)

/Users/childrebelsoldier/suppression/.venv/lib/python3.11/site-packages/transformer_lens/config/hooked_transformer_config.py:354: UserWarning: MPS backend may produce silently incorrect results (PyTorch 2.12.1). Set TRANSFORMERLENS_ALLOW_MPS=1 to suppress this warning. See: https://github.com/TransformerLensOrg/TransformerLens/issues/1178
  warn_if_mps(self.device)


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 1/338 [00:04<22:41,  4.04s/it]

Loading weights:  51%|█████     | 171/338 [00:04<00:03, 54.19it/s]

Loading weights:  60%|██████    | 203/338 [00:04<00:02, 54.90it/s]

...

Loading weights:  93%|█████████▎| 315/338 [00:07<00:00, 43.70it/s]

Loading weights:  97%|█████████▋| 327/338 [00:07<00:00, 44.16it/s]

Loading weights: 100%|██████████| 338/338 [00:07<00:00, 44.74it/s]

Loaded pretrained model Qwen/Qwen2.5-1.5B-Instruct into HookedTransformer


loaded Qwen/Qwen2.5-1.5B-Instruct | layers: 28 | d_model: 1536 | device: mps


In [4]:
# --- run all MVP behaviors via the shared pipeline (src/pipeline.py); retain acts/masks so the
#     robustness + matched_output contrasts can be re-derived without another GPU pass ---
signal_dirs, sup_dirs, checks, acts_by, masks_by = {}, {}, {}, {}, {}
for b in SUBSET:
    print("extracting:", b)
    ex = PL.extract_behavior(bundle, b, cfg, layer=LAYER, position=POS, contrast_mode=CONTRAST_MODE,
                             max_items=MAX_PAIRS, max_new_tokens=MAX_NEW_TOKENS)
    signal_dirs[b], sup_dirs[b], checks[b] = ex.signal_dir, ex.suppression_dir, ex.override_check
    acts_by[b], masks_by[b] = ex.acts, ex.masks
    print(f"  override_check = {ex.override_check}")
    M.free()

admitted = [b for b in SUBSET if checks[b].get("admits")]
print("\nadmitted to the family:", admitted)

extracting: sycophancy


  override_check = {'signal_probe_auc': 1.0, 'output_flip_rate': 0.0, 'admits': False, 'thresholds': {'auc': 0.7, 'flip': 0.5}}
extracting: safety


  override_check = {'signal_probe_auc': 1.0, 'output_flip_rate': 1.0, 'admits': True, 'thresholds': {'auc': 0.7, 'flip': 0.5}}
extracting: knowledge_conflict


  override_check = {'signal_probe_auc': 1.0, 'output_flip_rate': 0.5, 'admits': False, 'thresholds': {'auc': 0.7, 'flip': 0.5}}

admitted to the family: ['safety']


In [5]:
# --- the decisive matrices: raw + residual suppression cosines ---
sig_list = [signal_dirs[b] for b in SUBSET]
sup_list = [sup_dirs[b] for b in SUBSET]

C_raw, labels = S.cosine_matrix(sup_list)
resid    = P.residualize_all(sup_list, sig_list, own_only=True)  # project out each behavior's signal
C_res, _ = S.cosine_matrix(resid)

print("labels:", labels)
print("raw cosines:\n", C_raw.numpy().round(3))
print("residual cosines:\n", C_res.numpy().round(3))

labels: ['sycophancy', 'safety', 'knowledge_conflict']
raw cosines:
 [[nan nan nan]
 [nan nan nan]
 [nan nan  1.]]
residual cosines:
 [[nan nan nan]
 [nan nan nan]
 [nan nan  1.]]


In [6]:
# --- plot ---
def heat(ax, C, labels, title):
    im = ax.imshow(C.numpy(), vmin=-1, vmax=1, cmap="RdBu_r")
    ax.set_xticks(range(len(labels))); ax.set_xticklabels(labels, rotation=45, ha="right")
    ax.set_yticks(range(len(labels))); ax.set_yticklabels(labels)
    for i in range(len(labels)):
        for j in range(len(labels)):
            ax.text(j, i, f"{C[i, j]:.2f}", ha="center", va="center", fontsize=9)
    ax.set_title(title); return im

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
heat(axes[0], C_raw, labels, "Suppression cosines (raw)")
im = heat(axes[1], C_res, labels, "Residual (signal projected out)")
fig.colorbar(im, ax=axes, fraction=0.025)
plt.savefig("../results/mvp_cosines.png", dpi=150, bbox_inches="tight")
plt.show()

/var/folders/32/tnpkklcx02xfmj8_wv0d4wph0000gn/T/ipykernel_4585/1146118289.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
# --- summarize + persist ---
summary = S.summarize(resid)
print("participation ratio (residual):", round(summary["participation_ratio"], 3))
print("PC1 variance explained        :", round(summary["shared_knob"]["pc1_var_explained"], 3))

np.savez("../results/mvp_cosines.npz",
         labels=np.array(labels), C_raw=C_raw.numpy(), C_res=C_res.numpy(),
         participation_ratio=summary["participation_ratio"],
         pc1_var=summary["shared_knob"]["pc1_var_explained"])
with open("../results/mvp_summary.json", "w") as f:
    json.dump({"labels": labels, "contrast_mode": CONTRAST_MODE, "behavioral_mode": BEHAVIORAL_MODE,
               "participation_ratio": summary["participation_ratio"],
               "pc1_var_explained": summary["shared_knob"]["pc1_var_explained"],
               "override_checks": {b: checks[b] for b in SUBSET}}, f, indent=2)
print("saved -> results/mvp_cosines.{png,npz}, results/mvp_summary.json")

_LinAlgError: linalg.svd: The algorithm failed to converge because the input matrix is ill-conditioned or has too many repeated singular values (error code: 2).

In [8]:
# --- pre-registered ROBUSTNESS: residual cosine structure under each contrast (no re-run) ---
# See docs/contrast-decision.md. Stable structure across contrasts = robust; a flip between
# stimulus contrasts (manip_vs_clean / paired) and held-fixed contrasts (overrode_vs_resisted /
# did) diagnoses manipulation-stimulus contamination. matched_output auto-skips (no 'legit' arm).
modes = [CONTRAST_MODE] + [m for m in cfg["contrast"].get("robustness_set", []) if m != CONTRAST_MODE]
eye = torch.eye(len(SUBSET), dtype=torch.bool)
for mode in modes:
    try:
        sup_m = [Dir.suppression_direction(acts_by[b], masks_by[b], mode=mode,
                                           layer=LAYER, position=POS, behavior=b) for b in SUBSET]
        res_m = P.residualize_all(sup_m, [signal_dirs[b] for b in SUBSET], own_only=True)
        Cm, _ = S.cosine_matrix(res_m)
        pr = S.participation_ratio(S.pca_spectrum(res_m)[0])
        print(f"  {mode:<26} residual off-diag: {Cm[~eye].numpy().round(3)}   PR={pr:.2f}")
    except Exception as e:
        print(f"  {mode:<26} skipped ({type(e).__name__}: {e})")

  overrode_vs_resisted       residual off-diag: [nan nan nan nan nan nan]   PR=nan
  did_overrode_vs_resisted   residual off-diag: [  nan   nan   nan 0.099   nan 0.099]   PR=nan
  manip_vs_clean             residual off-diag: [  nan   nan   nan 0.269   nan 0.269]   PR=nan
  paired_manip_minus_clean   residual off-diag: [ nan  nan  nan 0.39  nan 0.39]   PR=nan


In [9]:
# --- matched_output VALIDATION (2509.21305: sycophantic vs genuine, same output/different cause) ---
# Does the primary contrast align with the clean same-output/different-cause direction? A high cosine
# means the primary direction captures genuine SUPPRESSION rather than manipulation-stimulus content.
# Only defined where a legitimate same-output twin exists (safety has none -> auto-skipped).
for b in SUBSET:
    if "legit" not in acts_by[b]:
        print(f"  {b:<20} no matched_output twin (expected for safety)"); continue
    mo = Dir.suppression_direction(acts_by[b], masks_by[b], mode="matched_output",
                                   layer=LAYER, position=POS, behavior=b)
    nlf = int(masks_by[b]["legit_followed"].sum())
    print(f"  {b:<20} cos(primary[{CONTRAST_MODE}], matched_output) = {sup_dirs[b].cosine(mo):+.3f}"
          f"   (legit_followed={nlf}, overrode={int(masks_by[b]['overrode'].sum())})")

  sycophancy           cos(primary[overrode_vs_resisted], matched_output) = +nan   (legit_followed=2, overrode=0)
  safety               no matched_output twin (expected for safety)
  knowledge_conflict   no matched_output twin (expected for safety)


## Reading the result (pre-registered outcomes)

Look at the **residual** matrix and the participation ratio:

- **One shared direction** — residual cosines near ±1, participation ratio ≈ 1, PC1 explains ~all variance.
- **Distinct directions** — residual cosines near 0, participation ratio ≈ N (=3).
- **Distinct-but-shared-knob** — raw/residual cosines low, but `shared_knob` shows one axis (PC1)
  aligning the set; the phase-2 subspace-removal + steering checks then confirm the knob is causal.

A null (distinct) is a **finding**, not a failure. The MVP settles this correlationally across a
real family; phase 2 (steering, subspace-removal, random baseline) makes it causal.